In [8]:
#ERA5 Parse Data
import cdsapi

#https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=download

dataset = "reanalysis-era5-single-levels-timeseries"
request = {
    "variable": [
        "2m_dewpoint_temperature",
        "10m_wind_gust_since_previous_post_processing",
        "mean_sea_level_pressure",
        "skin_temperature",
        "surface_pressure",
        "surface_solar_radiation_downwards",
        "sea_surface_temperature",
        "surface_thermal_radiation_downwards",
        "2m_temperature",
        "total_precipitation",
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
        "100m_u_component_of_wind",
        "100m_v_component_of_wind",
        "mean_wave_direction",
        "mean_wave_period",
        "significant_height_of_combined_wind_waves_and_swell"
    ],
    "location": {"longitude": 48, "latitude": 20.5},
    "date": ["1940-01-01/2026-02-07"],
    "data_format": "netcdf"
 #   "data_format": "csv"
}

# client = cdsapi.Client()
# client.retrieve(dataset, request).download("era5_data.nc.gz")



2026-02-12 17:35:13,225 WARNING [2025-03-17T00:00:00] Please be aware that the generation of this dataset is using an alternative source for the ERA5 data and may be subject to changes over time (e.g. file format, data file structure, deprecation etc). This dataset should therefore be regarded as “experimental” and is **not recommended for use in a production environment**. 

Notification of changes via this catalogue entry banner and/or in the [Forum](https://forum.ecmwf.int/) will be provided on best efforts.
2026-02-12 17:35:13,227 INFO Request ID is 0aa5d975-7c2f-4e88-b575-1555f1a2667e
2026-02-12 17:35:13,295 INFO status has been updated to accepted
2026-02-12 17:35:22,008 INFO status has been updated to running
2026-02-12 17:36:03,540 INFO status has been updated to successful
                                                                                          

'era5_data.csv'

In [22]:
# 2. NetCDF beolvasása 
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine
import pandas as pd
import shutil
import xarray as xr
import zipfile

load_dotenv() # betölti a .env fájlt

db_host = os.getenv("DB_HOST")
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_name = os.getenv("DB_NAME")


engine = create_engine( f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}")

zip_path = "era5_data.zip"
extract_dir = "extracted"

# 1. ZIP kibontása
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

# 2. Összegyűjtjük az összes .nc fájlt
nc_files = [os.path.join(extract_dir, f) for f in os.listdir(extract_dir) if f.endswith(".nc")]

if not nc_files:
    raise FileNotFoundError("Nem található .nc fájl a ZIP-ben!")

print("Megtalált NetCDF fájlok:")
for f in nc_files:
    print(" -", f)

# 3. Mindegyiket beolvassuk és DataFrame-re alakítjuk
dfs = []
for nc in nc_files:
    print("Beolvasás:", nc)
    ds = xr.open_dataset(nc)
    df = ds.to_dataframe().reset_index()
    dfs.append(df)

# 3. NetCDF-ek összeolvasztása
datasets = [xr.open_dataset(nc) for nc in nc_files]
ds_merged = xr.merge(datasets, compat="override")

# 4. DataFrame konverzió
df = ds_merged.to_dataframe().reset_index()

# 5. Típusok normalizálása
df = df.convert_dtypes()
df = df.where(df.notnull(), None)

# 6. Betöltés MySQL-be
df.to_sql("era5_adatok", engine, if_exists="replace", index=False, chunksize=5000)

print("Sikeresen betöltve!")


Megtalált NetCDF fájlok:
 - extracted/reanalysis-era5-single-levels-timeseries-sfcy6vefdhh.nc
 - extracted/reanalysis-era5-single-levels-timeseries-wavksszd0vt.nc
Beolvasás: extracted/reanalysis-era5-single-levels-timeseries-sfcy6vefdhh.nc
Beolvasás: extracted/reanalysis-era5-single-levels-timeseries-wavksszd0vt.nc
Sikeresen betöltve!
